Complete the exercises below For **Assignment #13**.

Load the `ISLR2` and the `tidymodels` packages.

In [1]:
library('ISLR2')
library('tidymodels')

── Attaching package

✔ broom        1.0.9     ✔ recipes      1.3.1
✔ dials        1.4.2     ✔ rsample      1.3.1
✔ dplyr        1.1.4     ✔ tailor       0.1.0
✔ ggplot2      3.5.2     ✔ tidyr        1.3.1
✔ infer        1.0.9     ✔ tune         2.0.0
✔ modeldata    1.5.1     ✔ workflows    1.3.0
✔ parsnip      1.3.3     ✔ workflowsets 1.1.1
✔ purrr        1.1.0     ✔ yardstick    1.3.2

── Conflicts ───────
✖ purrr::discard() masks scales::discard()
✖ dplyr::filter()  masks stats::filter()
✖ dplyr::lag()     masks stats::lag()
✖ recipes::step()  masks stats::step()



In this assignment we will use the `Default` dataset which includes the default status for credit card customers (`default` variable) in addition to each customer's:

1. credit card balance (`balance` variable),
1. student status (`student` variable), and,
1. income (`income` variable).

In [2]:
Default |> head()

,default,student,balance,income
,<fct>,<fct>,<dbl>,<dbl>
1,No,No,729.5265,44361.625
2,No,Yes,817.1804,12106.135
3,No,No,1073.5492,31767.139
4,No,No,529.2506,35704.494
5,No,No,785.6559,38463.496
6,No,Yes,919.5885,7491.559


We will be modeling `default` with the customer features.

Before we begin let's count how many customers fall into each `default` category.

In [3]:
Default |> count(default)

default,n
<fct>,<int>
No,9667
Yes,333


The data is quite imbalanced. This will be important to keep in mind when we evaluate the performance of our model later. 

Run the code below to create and training data from `Default`. We will use the "test" dataset at the end to get a final evaluation of our best model's accuracy.

In [4]:
Default_split = initial_split(Default, prop = 0.90, strata = default)

Default_train = training(Default_split)
Default_test = testing(Default_split)

Create a logistic regression model called `mod`. Set the engine to `glm` and the mode to `classification`. 

In [8]:
mod = logistic_reg() |> 
    set_engine('glm') |>
    set_mode('classification')    

mod

Logistic Regression Model Specification (classification)

Computational engine: glm 


Our data is imbalanced. As such, a naive model that *always* predicts a customer to **not default** would be correct quite often. Let's start by calculating the "accuracy" of a naive model. This will be the baseline accuracy by which we evaluate other models.

In [5]:
# This code calculates the accuracy of a model that always predicts default to be "No"

Default_train |>
    mutate(.pred_naive = factor('No', levels = c('No', 'Yes'))) |>
    accuracy(truth = default, .pred_naive)

.metric,.estimator,.estimate
<chr>,<chr>,<dbl>
accuracy,binary,0.9661111


Let's use k-fold cross validation to evaluate the performance of a model where the outcome is `default` and the predictors are `income` and `balance`.

To start, use `vfold_cv` to generate 10 validation folds (i.e. set the `v` variable to 10). Set the `strata` argument to `default` so we preserve the distribution of `default` values in each fold.

Creat your folds below and use `glimpse` to look at the output table. Call your output folds tables "folds".

In [9]:
folds = vfold_cv(Default_train, v = 10, strata = default)

folds |> glimpse()

Rows: 10
Columns: 2
$ splits <list> [<vfold_split[8100 x 900 x 9000 x 4]>], [<vfold_split[8100 x 9…
$ id     <chr> "Fold01", "Fold02", "Fold03", "Fold04", "Fold05", "Fold06", "Fo…


The code below fits a model to each of your 10 folds. `collect_metrics` finds the average of evaluation metrics for each of your ten models. 

In [16]:
mod |> 
    fit_resamples(default ~ income + balance, folds) |>
    collect_metrics()

.metric,.estimator,mean,n,std_err,.config
<chr>,<chr>,<dbl>,<int>,<dbl>,<chr>
accuracy,binary,0.9724444,10,0.0013435820,pre0_mod0_post0
brier_class,binary,0.0219278,10,0.0007845523,pre0_mod0_post0
roc_auc,binary,0.9484550,10,0.0044026257,pre0_mod0_post0


❓How does the model accuracy compare to the naive model from above?

The mean model accuracy (accuracy = 0.9724444) is slighly higher than the naive model above (accuracy = 0.9661111).

Complete the cell below to evaluate a model also includes the `student` variable as as predictor.
1. use `default ~ income + balance + student` as the formula,
2. encode your `student` variable with `step_dummy`, and,
3. don't forget to `prep` your recipe!

In [18]:
rec = recipe(default ~ income + balance + student, data = Default_train) |>
    step_dummy(student)
# workflow did not accept prepped recipe

mod |>
    fit_resamples(rec, folds) |>
    collect_metrics()

.metric,.estimator,mean,n,std_err,.config
<chr>,<chr>,<dbl>,<int>,<dbl>,<chr>
accuracy,binary,0.97244444,10,0.0012372810,pre0_mod0_post0
brier_class,binary,0.02181687,10,0.0007931043,pre0_mod0_post0
roc_auc,binary,0.94904120,10,0.0045488180,pre0_mod0_post0


❓Does it appear that the model that includes `student` improves upon the first model with only `income` and `balance` as predictors?

No, the accuracy of the model with the student predictor is the same as the accuracy of the model without the student predictor.

Finally, estimate the accuracy of an `default ~ income + balance` model on the test data, `Default_test`. 

❓Does our model outperform a naive model?

In [23]:
mod_fit = mod |> fit(default ~ income + balance, Default_test)

mod_fit

parsnip model object


Call:  stats::glm(formula = default ~ income + balance, family = stats::binomial, 
    data = data)

Coefficients:
(Intercept)       income      balance  
 -1.375e+01    5.983e-05    5.896e-03  

Degrees of Freedom: 999 Total (i.e. Null);  997 Residual
Null Deviance:	    255.4 
Residual Deviance: 128.8 	AIC: 134.8

In [25]:
augment(mod_fit, Default_test) |>
    accuracy(truth = default, .pred_class)

.metric,.estimator,.estimate
<chr>,<chr>,<dbl>
accuracy,binary,0.983


Yes, our model outperforms the naive model. The accuracy of our model is 0.983, which is higher than the accuracy of the naive model, 0.9661111.